In [57]:
import torch
import torch.nn as nn
import torch.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torch.optim as optim
from torchvision import transforms


In [58]:
#Datasets applied under the transforms.functional.pil_to_tensor according to docs.pytorch.org
#Transforms from image to tensor
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(device)
data_train = torchvision.datasets.EMNIST(root = "data", split = "letters", train = True, download = True, transform = transforms.Compose([transforms.functional.pil_to_tensor, transforms.ConvertImageDtype(torch.float32)]), target_transform=lambda y: y - 1)
data_test = torchvision.datasets.EMNIST(root = "data", split = "letters", train = False, download = True, transform = transforms.Compose([transforms.functional.pil_to_tensor, transforms.ConvertImageDtype(torch.float32)]), target_transform= lambda y: y - 1)


mps


In [59]:
#Data Loaders + Batching Size
dataL_train = DataLoader(dataset = data_train, batch_size = 128, shuffle = True)
dataL_test = DataLoader(dataset = data_test, batch_size = 128, shuffle = True)

In [60]:
class CNN(nn.Module):
    
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList()
        print(type(self.layers))
    def forward(self, input):
        for layer in self.layers:
            input = layer(input)
        return input
    def add_layer(self, F):
        try:
            self.layers.append(F.to(device))
            return True
        except:
            print("Pass in a valid module")
            return False

In [75]:
cnn = CNN()
cnn.add_layer(nn.Conv2d(1, 8, 3))
cnn.add_layer(nn.ReLU())
cnn.add_layer(nn.MaxPool2d(2))
cnn.add_layer(nn.Conv2d(8,16,3))
cnn.add_layer(nn.ReLU())
cnn.add_layer(nn.MaxPool2d(2))
cnn.add_layer(nn.Flatten())
cnn.add_layer(nn.Linear(16*5*5, 128))
cnn.add_layer(nn.ReLU())
cnn.add_layer(nn.Linear(128, 26))

epochs = 20
opt = optim.SGD(cnn.parameters(), lr = 0.01)
CLR = optim.lr_scheduler.CosineAnnealingLR(optimizer =opt, T_max =epochs, eta_min = 0.000001)
criterion = nn.CrossEntropyLoss()
try:
    print("loading weights")
    cnn.load_state_dict(torch.load("cnn_weights.pt"))
    print("success loaded weights!")
except:
    print("No weights")

#argmax returns indicies
for epoch in range(epochs):
    train_loss = 0
    total_batches = 0
    for data in dataL_train:
        opt.zero_grad()
        val, label = data
        val = val.to(device)
        label = label.to(device)
        x = cnn.forward(val)
        loss = criterion(x, label)
        train_loss += loss.item()
        total_batches+=1
        loss.backward()
        opt.step()
    CLR.step()
    test_loss = 0
    test_batches = 0
    hits = 0
    for data in dataL_test:
        val, label = data
        val = val.to(device)
        label = label.to(device)
        x = cnn.forward(val)
        loss = criterion(x, label)
        test_loss += loss.item()
        print(x.shape)
        test_batches+=1

    print(f"Training Loss: {train_loss/total_batches}")
    print("=========================")
    print(f"Testing Loss: {test_loss/test_batches}")


torch.save(cnn.state_dict(), "cnn_weights.pt")

        
        

<class 'torch.nn.modules.container.ModuleList'>
loading weights
success loaded weights!
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size([128, 26])
torch.Size

KeyboardInterrupt: 